# 01, Data preparation

**Inputs:**
- `data/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv.zip`
- `data/telecom_supplement.csv` (generated by `scripts/generate_telecom_supplement.py`, Bitext
  has zero coverage of `network_operations` / `field_service`, so a classifier trained on Bitext
  alone can never predict those two departments)
- `config/department_map.yaml`

**Outputs:**
- `data/raw/*.csv` (extracted Bitext CSV)
- `data/processed/{train,val,test}.csv`
- EDA printed below

In [1]:
import os, sys
from pathlib import Path

# Notebook hygiene: always run from the v1 root, never hard-code absolute d:\... paths.
_here = Path.cwd()
_v1_root = _here if (_here / "pyproject.toml").exists() else _here.parents[0]
os.chdir(_v1_root)
sys.path.insert(0, str(_v1_root))

# Notebooks that touch libs.platform.registry/config must pin a profile explicitly, the
# process env has no APP_PROFILE by default, which silently falls back to "cpu" (real Ollama
# calls, no stub) and produces confusing failures/latency. Pin stub here; override the env var
# before importing libs.platform.config in the rare notebook that needs otherwise.
os.environ.setdefault("APP_PROFILE", "stub")
print("cwd:", Path.cwd(), "| APP_PROFILE:", os.environ["APP_PROFILE"])

cwd: D:\DSEP22\v1 | APP_PROFILE: stub


In [2]:
import re
import zipfile

import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

RANDOM_STATE = 22

## Extract the Bitext zip to `data/raw/`

In [3]:
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

bitext_zip = Path("../data/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv.zip")
with zipfile.ZipFile(bitext_zip) as zf:
    zf.extractall(raw_dir)

bitext_csv = next(raw_dir.glob("*.csv"))
print("extracted:", bitext_csv)

extracted: data\raw\Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv


## Load Bitext + the telecom supplement, EDA on categories/intents

In [4]:
bitext = pd.read_csv(bitext_csv)
telecom = pd.read_csv("data/telecom_supplement.csv")
print("bitext rows:", len(bitext), "| categories:", bitext["category"].nunique(), "| intents:", bitext["intent"].nunique())
print("telecom supplement rows:", len(telecom), "| categories:", sorted(telecom["category"].unique()))
bitext["category"].value_counts()

bitext rows: 26872 | categories: 11 | intents: 27
telecom supplement rows: 410 | categories: ['FIELDSERVICE', 'NETWORK']


category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64

In [5]:
df = pd.concat([bitext, telecom], ignore_index=True)
df.head(3)

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...


## Map Bitext `category` -> project `Department`

`ORDER,SHIPPING,DELIVERY -> general`; `REFUND,INVOICE,PAYMENT -> billing`;
`CONTACT,FEEDBACK -> general`; `ACCOUNT,SUBSCRIPTION -> technical_support`; `CANCEL -> retention`;
plus the telecom supplement's `NETWORK -> network_operations`, `FIELDSERVICE -> field_service`.
The mapping is data (`config/department_map.yaml`), not code, so it can be revised without a
notebook change.

In [6]:
dept_map = yaml.safe_load(Path("config/department_map.yaml").read_text(encoding="utf-8"))["category_to_department"]
df["department"] = df["category"].map(dept_map)
assert df["department"].isna().sum() == 0, "unmapped category found"
df["department"].value_counts()

department
general               11948
billing                6989
technical_support      6985
retention               950
network_operations      250
field_service           160
Name: count, dtype: int64

## Strip templated slots

The `instruction` column contains templated slots like `{{Order Number}}`; if left in, the
classifier learns to key off the literal `{{` / `}}` characters instead of the surrounding
language. Strip them (not randomise, the surrounding phrasing already carries the signal).

In [7]:
slot_re = re.compile(r"\{\{[^}]*\}\}")
df["text"] = df["instruction"].apply(lambda s: slot_re.sub("", s).strip())
df["text"] = df["text"].str.replace(r"\s+", " ", regex=True)
(df["instruction"].str.contains(slot_re)).sum(), (df["text"].str.contains(slot_re)).sum()

(np.int64(6670), np.int64(0))

## Stratified 80/10/10 split by department

In [8]:
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["department"], random_state=RANDOM_STATE)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["department"], random_state=RANDOM_STATE)

out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)
train_df[["text", "department", "category", "intent"]].to_csv(out_dir / "train.csv", index=False)
val_df[["text", "department", "category", "intent"]].to_csv(out_dir / "val.csv", index=False)
test_df[["text", "department", "category", "intent"]].to_csv(out_dir / "test.csv", index=False)

len(train_df), len(val_df), len(test_df)

(21825, 2728, 2729)

In [9]:
import json

metrics_path = Path("evaluation/reports/metrics.jsonl")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "a", encoding="utf-8") as f:
    f.write(json.dumps({
        "notebook": "01_data_preparation",
        "rows_total": len(df),
        "rows_train": len(train_df),
        "rows_val": len(val_df),
        "rows_test": len(test_df),
        "departments": df["department"].value_counts().to_dict(),
    }) + "\n")
print("appended metrics row")

appended metrics row
